### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 6_offset_computation_leg2
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es)
### Data from FaSt-SWOT experiment
# 
**DESCRIPTION**:
Direct visualization of salinity differences to determine the true offset needed. Compares step5 MVP data (without offset) against CTD reference measurements for Leg 2 (deeper profiles).
#
INPUT: Step5 MVP NetCDF files and CTD .cnv reference files
#
OUTPUT: Offset difference plots (*.png) showing MVP-CTD salinity profiles
### ==============================================================================

**PURPOSE**: Quantify the offset by comparing step5 MVP data (WITHOUT offset) against CTD reference.

In [1]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from geopy.distance import geodesic
import gsw
import re
import io
import warnings

warnings.filterwarnings('ignore')

BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")
DIR_MVP_RAW = BASE_ROOT / "Data" / "Leg2" / "processed_step1_highres_qc"
DIR_MVP_FINAL = BASE_ROOT / "Data" / "Leg2" / "processed_step5"
DIR_CTD = Path(r"C:/Users/ASUS/OneDrive - Universitat de les Illes Balears/SWOT/CTD/CTD_data/leg2/HM/")
# OUTDIR = BASE_ROOT / "OFFSET_DIFFERENCE_PLOTS_LEG2"
# OUTDIR.mkdir(parents=True, exist_ok=True)

MAX_DIST_KM = 0.5
MAX_TIME_MIN = 60.0
Z_MAX = 200.0  # Leg 2 is deeper
CTD_SMOOTH_WIN = 8

print(f"Processing Leg 2 offset quantification...")
print(f"MVP final data: {DIR_MVP_FINAL}")
print(f"CTD data: {DIR_CTD}")
# print(f"Output: {OUTDIR}\n")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Processing Leg 2 offset quantification...
MVP final data: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg2\processed_step5
CTD data: C:\Users\ASUS\OneDrive - Universitat de les Illes Balears\SWOT\CTD\CTD_data\leg2\HM


In [2]:
# Functions
def read_ctd_cnv(path):
    """Read CTD .cnv file"""
    try:
        with open(path, 'r', encoding='latin-1') as f:
            lines = f.readlines()
        lat, lon, time_val, start_idx = np.nan, np.nan, pd.NaT, 0
        col_map = {}
        for i, line in enumerate(lines):
            if 'NMEA Latitude' in line:
                p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(p) >= 2:
                    lat = float(p[0]) + float(p[1])/60 * (-1 if 'S' in line else 1)
            if 'NMEA Longitude' in line:
                p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(p) >= 2:
                    lon = float(p[0]) + float(p[1])/60 * (-1 if 'W' in line else 1)
            if 'start_time' in line:
                try:
                    time_val = pd.to_datetime(line.split('=')[1].strip().split('[')[0])
                except:
                    pass
            if '# name' in line:
                idx = int(re.search(r"name (\d+)", line).group(1))
                name = line.split('=')[1].split(':')[0].strip().lower()
                if name in ('prdm', 'pres', 'pressure'):
                    col_map['p'] = idx
                elif name in ('t090c', 't190c', 'temp', 'tv290c'):
                    col_map['t'] = idx
                elif name in ('sal00', 'sal11', 'sal', 'psal'):
                    col_map['s'] = idx
            if '*END*' in line:
                start_idx = i + 1
                break
        df = pd.read_csv(io.StringIO(''.join(lines[start_idx:])), delim_whitespace=True, header=None, on_bad_lines='skip')
        p = df.iloc[:, col_map['p']].values
        t = df.iloc[:, col_map['t']].values
        s = df.iloc[:, col_map['s']].values
        return {'lat': lat, 'lon': lon, 'time': time_val, 'id': path.stem, 'p': p, 't': t, 's': s}
    except:
        return None

def load_mvp_catalog(mvp_dir):
    """Load MVP file catalog"""
    data = []
    for f in sorted(mvp_dir.glob('*.nc')):
        try:
            with xr.open_dataset(f) as ds:
                lat, lon = np.nan, np.nan
                for ln in ('latitude', 'lat'):
                    if ln in ds:
                        lat = float(ds[ln].values.flatten()[0])
                        break
                for ln in ('longitude', 'lon'):
                    if ln in ds:
                        lon = float(ds[ln].values.flatten()[0])
                        break
                t_val = pd.to_datetime(ds.time.values.flatten()[0]) if 'time' in ds else pd.NaT
                if np.isfinite(lat) and np.isfinite(lon):
                    data.append({'file': f.name, 'lat': lat, 'lon': lon, 'time': t_val, 'path': f})
        except:
            pass
    return pd.DataFrame(data)

def find_closest_mvp(ctd, df_mvp):
    """Find closest MVP to CTD"""
    if df_mvp.empty or not (np.isfinite(ctd['lat']) and np.isfinite(ctd['lon'])):
        return None, np.nan, np.nan
    dists = np.array([geodesic((ctd['lat'], ctd['lon']), (r.lat, r.lon)).km for r in df_mvp.itertuples()])
    best_idx = int(np.argmin(dists))
    best_row = df_mvp.iloc[best_idx]
    dist_km = dists[best_idx]
    dt_min = abs((best_row['time'] - ctd['time']).total_seconds() / 60.0) if pd.notna(best_row['time']) and pd.notna(ctd['time']) else 999.0
    if dist_km <= MAX_DIST_KM and dt_min <= MAX_TIME_MIN:
        return best_row, dist_km, dt_min
    return None, dist_km, dt_min

print('Loading catalogs...')
df_mvp = load_mvp_catalog(DIR_MVP_RAW)
ctd_files = sorted(DIR_CTD.glob('d*.cnv'))
print(f'Found {len(df_mvp)} MVP profiles and {len(ctd_files)} CTD files.\n')

Loading catalogs...
Found 297 MVP profiles and 10 CTD files.



In [3]:
# Process each CTD station
offsets_all = []

for ctd_f in ctd_files:
    ctd = read_ctd_cnv(ctd_f)
    if ctd is None:
        continue

    best_mvp, dist_km, dt_min = find_closest_mvp(ctd, df_mvp)
    if best_mvp is None:
        continue

    f_final = DIR_MVP_FINAL / best_mvp['file'].replace('_step1_qc.nc', '_step5.nc')
    if not f_final.exists():
        continue

    try:
        with xr.open_dataset(best_mvp['path']) as ds_raw, xr.open_dataset(f_final) as ds_f:
            p_mvp = ds_raw.pressure.values
            t_raw = ds_raw.t1.values
            c_raw = ds_raw.c1.values
            if np.nanmean(c_raw) < 10:
                c_raw = c_raw * 10
            s_raw = gsw.SP_from_C(c_raw, t_raw, p_mvp)
            
            s_final = ds_f.salinity_practical.values  # NO offset
            t_final = ds_f.t1_smooth.values

        # Smooth CTD
        s_ctd_smooth = pd.Series(ctd['s']).rolling(CTD_SMOOTH_WIN, center=True, min_periods=1).mean().values

        # Common grid
        p_grid = np.arange(10, Z_MAX, 0.5)
        s_ctd_i = np.interp(p_grid, ctd['p'], s_ctd_smooth)
        s_final_i = np.interp(p_grid, p_mvp, s_final)
        
        # Calculate difference (MVP - CTD)
        diff = s_final_i - s_ctd_i
        mask = np.isfinite(diff)
        offset_median = np.nanmedian(diff[mask])
        
        offsets_all.append({
            'ctd_id': ctd['id'],
            'p_grid': p_grid[mask],
            'diff': diff[mask],
            'offset_median': offset_median
        })

        # # PLOT DIFFERENCE
        # fig, ax = plt.subplots(figsize=(10, 8))
        # ax.plot(diff, p_grid, 'b-', lw=2, label='MVP - CTD (current)')
        # ax.axvline(x=0, color='k', linestyle='--', alpha=0.5, label='Zero offset')
        # ax.axvline(x=offset_median, color='r', linestyle='--', lw=2, label=f'Median offset: {offset_median:.4f} PSU')
        # ax.set_xlabel('Salinity Difference (PSU)', fontsize=12)
        # ax.set_ylabel('Pressure (dbar)', fontsize=12)
        # ax.set_title(f'MVP-CTD Salinity Difference: {ctd["id"]} (Dist: {dist_km:.2f} km, Dt: {dt_min:.0f} min)', fontsize=13, fontweight='bold')
        # ax.invert_yaxis()
        # ax.grid(True, alpha=0.3)
        # ax.legend(fontsize=11)
        # plt.tight_layout()
        # fig.savefig(OUTDIR / f'Offset_Diff_{ctd["id"]}.png', dpi=150)
        # plt.close()
        
        print(f'  OK: {ctd["id"]:12s} offset_median = {offset_median:+.4f} PSU')

    except Exception as e:
        print(f'  ERROR {ctd["id"]}: {e}')

# Summary
if offsets_all:
    print('\n' + '='*70)
    print('SUMMARY - MVP(step5) - CTD Differences (Leg 2):')
    print('='*70)
    medians = [x['offset_median'] for x in offsets_all]
    mean_offset = np.mean(medians)
    median_offset = np.median(medians)
    min_offset = np.min(medians)
    max_offset = np.max(medians)
    
    print(f'  Number of stations: {len(medians)}')
    print(f'  Mean offset:   {mean_offset:+.4f} PSU')
    print(f'  Median offset: {median_offset:+.4f} PSU')
    print(f'  Min offset:    {min_offset:+.4f} PSU')
    print(f'  Max offset:    {max_offset:+.4f} PSU')
    print('\n  RECOMMENDATION:')
    print(f'  Apply offset = {-median_offset:+.4f} PSU (negative of median difference)')
    print('\n  This offset will be applied in step 6 to correct the MVP data')
    # print(f'\nPlots saved to: {OUTDIR}')
else:
    print('ERROR: No valid matches found')

  OK: ds2-03       offset_median = +0.0219 PSU
  OK: ds4-04       offset_median = +0.0209 PSU
  OK: ds4-06       offset_median = +0.0099 PSU
  OK: ds5-09       offset_median = +0.0303 PSU

SUMMARY - MVP(step5) - CTD Differences (Leg 2):
  Number of stations: 4
  Mean offset:   +0.0207 PSU
  Median offset: +0.0214 PSU
  Min offset:    +0.0099 PSU
  Max offset:    +0.0303 PSU

  RECOMMENDATION:
  Apply offset = -0.0214 PSU (negative of median difference)

  This offset will be applied in step 6 to correct the MVP data
